#  Reflexión final

##### a) **¿Por qué la reconstrucción tomográfica se considera un problema inverso?**

La reconstrucción tomográfica es un **problema inverso** porque buscamos recuperar una imagen desconocida $x$ a partir de información conocida:

- Las **mediciones $b$**, que representan la atenuación acumulada a lo largo de cada rayo.
- La **matriz de proyección $A$**, que describe cómo los rayos atraviesan los píxeles.

El modelo $b = Ax$ relaciona la imagen con las mediciones: cada componente de $b$ es una suma ponderada de las intensidades de los píxeles atravesados por un rayo.

Mientras que el **problema directo** calcula las mediciones a partir de una imagen conocida, el **problema inverso** busca recuperar la imagen a partir de esas mediciones.
```


##### b) **¿Cómo puede determinarse, a partir de A, si las mediciones permiten recuperar una única imagen?**
La unicidad depende del **rango** y del **espacio nulo** de la matriz $A$.

Si $A$ tiene $n$ columnas y $\operatorname{rango}(A)=n$, hay un pivote en cada columna y su espacio nulo es $\operatorname{Nul}(A)=\{0\}.$

Por lo tanto, si el sistema $Ax=b$ tiene solución, esta es única.

En cambio, si $\operatorname{rango}(A)<n$, existen variables libres y algún vector $v\neq 0$ tal que $Av=0$. Si $x$ es una solución, entonces:

$$
A(x+tv)=Ax+tAv=b
\qquad \text{para todo } t\in\mathbb{R}.
$$

Así, **distintas imágenes producen las mismas mediciones**, por lo que, si existe una solución, hay infinitas soluciones del sistema. El espacio nulo representa los cambios en la imagen que las mediciones no pueden detectar.


##### c) **¿Cómo afecta el ruido al sistema de ecuaciones?**

El ruido altera las mediciones, de modo que el sistema pasa a ser:

$$
b_\sigma = Ax_{\text{original}} + \eta,
$$

donde $\eta$ representa el ruido. Esto puede hacer que el sistema sea incompatible, es decir, que no exista una imagen que reproduzca exactamente todas las mediciones.

Con mínimos cuadrados buscamos una reconstrucción que minimice la diferencia entre las mediciones calculadas y las observadas:

$$
\hat{x} = \arg\min_x \|Ax-b_\sigma\|_2^2.
$$

El residuo $r=A\hat{x}-b_\sigma$ representa esa diferencia. Su norma indica qué tan bien ajustamos las mediciones, pero **no es suficiente para evaluar la calidad de la reconstrucción**, porque un buen ajuste también puede reproducir el ruido.

En el ejercicio 9 observamos que, sin regularización y con ruido, el error respecto del fantoma podía aumentar aunque el costo siguiera disminuyendo. Al incorporar el término $\lambda\|Dx\|_2^2$, penalizamos las diferencias entre píxeles vecinos y favorecimos reconstrucciones más suaves. Sin embargo, una regularización excesiva también podía borrar detalles.

Por eso, evaluamos las reconstrucciones utilizando tanto el error relativo respecto del fantoma conocido como las visualizaciones, además de la norma del residuo.

##### d) **¿Qué supuesto sobre la imagen introduce el término de regularización?**

El término $\|Dx\|_2^2$ introduce el supuesto de que **los píxeles vecinos tienden a tener intensidades similares**, por lo que favorece imágenes suaves. Como $D$ calcula diferencias entre píxeles vecinos, este término aumenta cuando esas diferencias son grandes.

La función que minimizamos es:

$$
J(x)=\frac{1}{2}\left(\|Ax-b_\sigma\|_2^2+\lambda\|Dx\|_2^2\right).
$$

El parámetro $\lambda$ controla la importancia de la suavidad frente al ajuste a las mediciones:

- **Si $\lambda=0$**, solo ajustamos las mediciones, lo que puede llevarnos a reproducir también el ruido.
- **Con un $\lambda$ moderado**, reducimos las variaciones producidas por el ruido, aunque también podemos suavizar algunos detalles.
- **Con un $\lambda$ demasiado grande**, penalizamos excesivamente las diferencias entre vecinos, Pero esa penalización no distingue entre ruido y bordes reales. Si aumentamos demasiado $\lambda$, también borramos detalles de la imagen.

En el ejercicio 9 obtuvimos el menor error final con $\lambda=0$ cuando no había ruido, con $\lambda=0.1$ para $\sigma=0.10$ y con $\lambda=0.3$ para $\sigma=0.20$. Esto muestra que, al aumentar el ruido, nos resultó conveniente dar más importancia a la regularización.

Sin ruido, el suavizado modificó detalles reales del fantoma y aumentó el error. Con ruido, una regularización moderada mejoró la reconstrucción al reducir sus efectos. Sin embargo, aumentar $\lambda$ no siempre mejoró el resultado: con $\lambda=1$ obtuvimos un error mayor que con el mejor valor de cada caso.

### e) **Comparación de los métodos**

Con `np.linalg.lstsq` obtuvimos la solución de mínimos cuadrados mediante una llamada a la función, sin elegir una tasa de aprendizaje, un punto inicial ni un criterio de parada. Si existen varias soluciones de mínimos cuadrados, devuelve la de menor norma.

El **gradiente descendiente** aproxima la solución mediante actualizaciones sucesivas:

$$
x_{k+1}=x_k-\alpha A^T(Ax_k-b).
$$

Requiere un punto inicial, una tasa de aprendizaje $\alpha$ y un criterio de parada, como una tolerancia y un máximo de iteraciones. Una tasa pequeña produce un avance lento, mientras que una demasiado grande puede impedir la convergencia. Cuando converge y la solución es única, se aproxima al mismo resultado que `np.linalg.lstsq`.

El **método de Newton** utiliza tanto el gradiente como la Hessiana. Para la función de mínimos cuadrados,

$$
J(x)=\frac12\|Ax-b\|_2^2,
\qquad
\nabla J(x)=A^T(Ax-b),
\qquad
H=A^TA.
$$

En cada iteración resolvemos:

$$
Hp_k=-\nabla J(x_k),
\qquad
x_{k+1}=x_k+p_k.
$$

Newton requiere un punto inicial y, en una implementación iterativa, una tolerancia y un máximo de iteraciones. En su versión con paso completo no necesitamos ajustar una tasa de aprendizaje.

Si $A$ tiene rango columna completo, $A^TA$ es invertible y un único paso de Newton da:

$$
x_{k+1}
=x_k-(A^TA)^{-1}(A^TAx_k-A^Tb)
=(A^TA)^{-1}A^Tb.
$$

Esto explica por qué Newton puede alcanzar la solución de mínimos cuadrados en una sola actualización, salvo errores de redondeo: la función es cuadrática y su Hessiana es constante. El programa puede realizar una comprobación adicional para detectar la convergencia.

Por lo tanto, para el mismo problema con solución única, los tres métodos deberían obtener reconstrucciones prácticamente iguales, con diferencias debidas a la tolerancia y al redondeo. Newton necesita menos iteraciones, pero cada paso requiere resolver un sistema lineal. Si $A$ no tiene rango columna completo, la Hessiana es singular y no podemos aplicar directamente esta versión de Newton.

### f) **Síntesis final**

A lo largo del trabajo utilizamos el modelo $b=Ax$ para relacionar la imagen desconocida $x$ con las mediciones $b$, mediante la matriz de proyección $A$. Los distintos conceptos nos permitieron analizar aspectos complementarios de la reconstrucción:

- **El rango de $A$** nos permitió determinar si las mediciones contienen suficiente información para recuperar una única imagen. Si hay un pivote en cada columna, la solución, cuando existe, es única.

- **El espacio nulo de $A$** describe las modificaciones de la imagen que las mediciones no detectan. Si $Av=0$ con $v\neq0$, entonces $x$ y $x+v$ producen las mismas mediciones, por lo que no podemos distinguirlas a partir de esos datos.

- **Mínimos cuadrados** nos permitió trabajar con mediciones ruidosas cuando no existe una solución exacta. Buscamos una imagen $\hat{x}$ que minimice $\|Ax-b\|_2^2$.

- **Las proyecciones ortogonales** explican geométricamente esa solución: $A\hat{x}$ es la proyección de $b$ sobre el espacio columna de $A$, formado por las mediciones que el modelo puede reproducir. El residuo es perpendicular a ese espacio:

$$
A^T(A\hat{x}-b)=0.
$$

- **La regularización** incorpora un supuesto adicional sobre la imagen para reducir el efecto del ruido. Al minimizar

$$
J(x)=\frac12\left(\|Ax-b\|_2^2+\lambda\|Dx\|_2^2\right),
$$

buscamos un equilibrio entre ajustar las mediciones y favorecer intensidades similares entre píxeles vecinos. El parámetro $\lambda$ controla ese equilibrio: un valor insuficiente puede permitir que ajustemos el ruido, mientras que uno excesivo puede borrar detalles.

Así, el rango y el espacio nulo nos ayudaron a analizar la unicidad; mínimos cuadrados y las proyecciones ortogonales, el ajuste a las mediciones; y la regularización, cómo incorporar información sobre la imagen para mejorar la reconstrucción. Los resultados también nos mostraron que un residuo pequeño no garantiza una imagen cercana a la original.